# Task 1.2 — From-scratch CNN encoder (Kaggle runner)

This notebook only **runs** the project code; all logic lives in `src/` and is called through the CLI.

**Before running**
1. *Settings → Accelerator*: **GPU T4 x1** (or P100). *Settings → Internet*: **on**.
2. *Add-ons → Secrets*: add `GITHUB_TOKEN` (GitHub token with read access to the repo) and `WANDB_API_KEY`, and attach both to this notebook.
3. Run **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed (~6–8 h in total, Kaggle limit is 12 h).

Everything written under `outputs/` is saved to `/kaggle/working/outputs` (kept as the notebook output).
If a step fails, the notebook stops there instead of running the remaining steps.

In [ ]:
import os
import shlex
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY").strip()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN").strip()

REPO = "omarsaher/image-classfication-and-caption"
BRANCH = "feature/scratch-cnn"
WORKDIR = "/tmp/repo"                      # code + data (not saved as notebook output)
OUTPUTS = "/kaggle/working/outputs"        # results (saved as notebook output)


def cli(args):
    """Run `python -m src.cli <args>`, show its output, and stop the notebook if it fails."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1"}
    with subprocess.Popen(["python", "-m", "src.cli", *shlex.split(args)], env=env, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode != 0:
        raise RuntimeError(f"Step failed (exit code {proc.returncode}): {args}")

In [ ]:
# Clone the private repo; the token is removed from the remote URL right after
if not os.path.exists(WORKDIR):
    result = subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH,
         f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", WORKDIR],
        capture_output=True, text=True,
    )
    print(result.stderr.replace(GITHUB_TOKEN, "***")[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"Clone failed (exit code {result.returncode}); check the GITHUB_TOKEN secret")
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)

os.makedirs(OUTPUTS, exist_ok=True)
if not os.path.exists(f"{WORKDIR}/outputs"):
    os.symlink(OUTPUTS, f"{WORKDIR}/outputs")
os.chdir(WORKDIR)
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Check the W&B key now, so a bad key stops the notebook before any training
import wandb

if not wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True, verify=True):
    raise RuntimeError("W&B login failed; check the WANDB_API_KEY secret")
print("W&B login OK")

## 0. Quick check (1 epoch, 5 batches, no tracking)

In [ ]:
cli("train --task classification --config configs/cls_scratch.yaml "
    "--set experiment.name=kaggle_smoke train.epochs=1 train.max_batches=5 tracking.backend=none")

## 1. Hyperparameter tuning — Grid Search vs Random Search (originals only)
12 trials × 12 epochs each. Results are saved after every trial.

In [ ]:
cli("tune --task classification --config configs/tune_scratch.yaml --method grid")

In [ ]:
cli("tune --task classification --config configs/tune_scratch.yaml --method random")

In [ ]:
cli("compare-tuning --task classification --results-dir outputs/tuning")

## 2. Final training with the best hyperparameters
- **Run A**: originals only
- **Run B**: originals + the team's augmented copies (only this flag changes)

In [ ]:
cli("train --task classification --config configs/cls_scratch.yaml --best-from outputs/tuning "
    "--set experiment.name=scratch_A_originals data.use_offline_augmented=false")

In [ ]:
cli("train --task classification --config configs/cls_scratch.yaml --best-from outputs/tuning "
    "--set experiment.name=scratch_B_augmented data.use_offline_augmented=true")

In [ ]:
cli("compare-runs --task classification "
    "--runs outputs/classification/scratch_A_originals outputs/classification/scratch_B_augmented "
    "--names 'A: originals' 'B: + augmented copies' "
    "--output outputs/classification/run_comparison.png")

## 3. Test-set evaluation (5,794 images) and Grad-CAM

In [ ]:
for run in ["scratch_A_originals", "scratch_B_augmented"]:
    cli(f"evaluate --task classification --checkpoint outputs/classification/{run}/best.pt --split test")

In [ ]:
for run in ["scratch_A_originals", "scratch_B_augmented"]:
    cli(f"gradcam --task classification --checkpoint outputs/classification/{run}/best.pt --split test")

## 4. Pack the results for download

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/task1_2_results", "zip", OUTPUTS)
!ls -la /kaggle/working